# 歌词采集-QQ音乐

In [1]:
import requests
import re
import json
import os
import time
import pandas as pd
import html
from datetime import datetime
from collections import defaultdict


from collections import Counter


In [2]:
import sys
sys.path.append('..')

# 通用方法

## 时间戳格式化

In [3]:
def format_timestamp(ts, date_format='%Y-%m-%d'):
    """
    自动识别秒或毫秒，并转换为指定格式的字符串
    :param ts: 时间戳 (int 或 float)
    """
    if not ts or ts <= 0:
        return "Unknown"
    
    # 核心逻辑：判断时间戳位数
    # 秒级时间戳目前在 10^9 数量级（10位）
    # 毫秒级时间戳在 10^12 数量级（13位）
    # 我们以 10^11 (11位) 为界限进行区分
    if ts > 100000000000: 
        ts = ts / 1000  # 是毫秒，转换为秒
    
    try:
        dt = datetime.fromtimestamp(ts)
        return dt.strftime(date_format)
    except Exception:
        return "Invalid Date"

## 增量保存到json文件

In [4]:
def save_to_json_list(file_path, song_data):
    """以列表形式保存所有歌曲，避免字典 key 覆盖的问题"""
    data_list = []
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            try:
                data_list = json.load(f)
                if not isinstance(data_list, list): data_list = []
            except:
                data_list = []

    data_list.append(song_data)

    with open(file_path, 'w', encoding='utf-8') as f:
        json.dump(data_list, f, ensure_ascii=False, indent=4)

# 按歌手采集曲目

In [5]:
def search_song(keyword, page=0):
    """搜索歌曲并返回歌曲ID"""
    url = "https://c.y.qq.com/soso/fcgi-bin/client_search_cp"
    params = {
        "w": keyword,
        "format": "json",
        "n": 50,
        "p": page,
    }
    headers = {
        "User-Agent":
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url, params=params, headers=headers)
    if response.status_code == 200:
        data = json.loads(response.text)
        songs = data["data"]["song"]["list"]
        res = []
        for song in songs:
            res_d = {
                "song_id": song["songid"],
                "song_mid": song["songmid"],
                "song_name": song["songname"],
                "song_subname": song["lyric"],
                "artist_name": ",".join([singer["name"] for singer in song["singer"]]),
                "artist_id": ",".join([str(singer["id"]) for singer in song["singer"]]),
                "artist_mid": ",".join([singer["mid"] for singer in song["singer"]]),
                "album_name": song['albumname'],
                "album_id": song['albumid'],
                "album_mid": song['albummid'],
                "duration": song['interval'],
                "publish_time": song["pubtime"],
            }
            res.append(res_d)
        return res
    return []

In [6]:
def get_songs_data_raw(singger, max_page=1):
    """
    获取歌手歌曲列表
    singer_name: 歌手名称
    max_page: 最大页数, 默认每页50条数据，max_page=曲目总数/50
    """
    qq_songs_list = []
    for page in range(0, max_page):
        print(f'正在获取第{page+1}页数据...')
        res = search_song(singger, page)
        time.sleep(2)
        qq_songs_list.extend(res)
    return qq_songs_list

# 曲目过滤

## OST曲目筛选

In [7]:
# 数据筛选
# 1. artist_name中包含singger
# 2. song_subname中包含书名号，将书名号中的内容保存为新字段: tv_name
def filter_ost_songs(singger, song_list):
    """
    筛选符合条件的歌曲：
    1. 歌手包含 singger
    2. 子标题包含书名号，并提取书名号内容为 tv_name
    """
    filtered_list = []
    # 预编译正则，匹配《 和 》之间最少的内容
    tv_pattern = re.compile(r'《(.*?)》')

    for song in song_list:
        # 条件 1: 校验 artist_name (确保该字段已在之前的解析中生成)
        if singger not in song.get("artist_name", ""):
            continue

        # 条件 2: 校验 song_name，如果”《“在 song_name 中，则跳过
        if "《" in song.get("song_name", ""):
            continue

        # 条件 2.5: 如果 song_subname 中包含 试听，伴奏，则跳过
        if any(keyword in song.get("song_name", "") for keyword in ["试听", "伴奏"]):
            continue
            
        # 条件 3: 校验 song_subname 并在满足时提取 tv_name
        subname = song.get("song_subname", "")
        match = tv_pattern.search(str(subname))
        
        if match:
            # 满足条件，创建新字段并保存
            song["tv_name"] = match.group(1)
            filtered_list.append(song)
            
    return filtered_list

## 按专辑列表

In [8]:
def filter_album_songs(singger, song_list, album_list):
    """
    筛选符合条件的歌曲：
    1. 歌手包含 singger
    2. 专辑包含在 album_list 中
    """
    filtered_list = []

    for song in song_list:
        singers = song.get("artist_name", "")
        album_name = song.get("album_name", "")
        if singger in singers and album_name in album_list:
            filtered_list.append(song)
            
    return filtered_list

## 曲目数据清洗

In [9]:
def clear_song_data(song_data,
                    is_filter_ost=False,
                    is_use_raw_song_name=False):
    """
    清洗歌曲数据
    is_filter_ost: 是否过滤掉OST歌曲
    is_use_raw_song_name: 二次清洗时，是否使用原始歌曲名，例如五月天这种，原始歌曲就有多个版本的，需要在歌名中保留括号内容
    """
    songs_df = pd.DataFrame(song_data)
    if is_use_raw_song_name:
        songs_df['song_name_unique'] = songs_df['song_name']
    else:
        # 分割song_name中的括号
        songs_df['song_name_unique'] = songs_df['song_name'].apply(
            lambda x: x.split('(')[0])
        songs_df['song_name_unique'] = songs_df['song_name_unique'].apply(
            lambda x: x.split('（')[0])
        # 删除前后空格
        songs_df['song_name_unique'] = songs_df['song_name_unique'].apply(
            lambda x: x.strip())
    # 按song_name_unique进行去重
    songs_df = songs_df.drop_duplicates(subset=['song_name_unique'],
                                        keep='first')
    # 发行时间格式化
    songs_df['publish_date'] = songs_df['publish_time'].apply(
        lambda x: format_timestamp(x))
    if is_filter_ost:
        # 二次筛选ost， 新建列 is_ost, 如果song_subname中含 影，剧，曲任意一个字，则is_ost为1，否则为0
        # songs_df['is_ost'] = songs_df['song_subname'].apply(
        #     lambda x: 1 if '影' in str(x) or '剧' in str(x) or '曲' in str(
        #         x) or '片' in str(x) else 0)
        songs_df['is_ost'] = songs_df['song_subname'].apply(
            lambda x: 1 if '影' in str(x) or '剧' in str(x) or '片' in str(x) else 0)
        songs_df = songs_df[songs_df['is_ost'] == 1]
        # songs_df = songs_df
    return songs_df

## 专辑信息清洗

In [10]:
# 专辑数据清洗
def clear_album_data(songs_data):
    album_count = songs_data.groupby(
        ['album_name',
         'album_id'])['album_id'].count().reset_index(name='count')
    album_count = album_count.sort_values(by=['count'], ascending=False)
    # 取count最大值所在行的数据为album_id
    album_id = album_count.drop_duplicates(subset=['album_name'],
                                           keep='first').reset_index(drop=True)
    # 匹配专辑发行日期
    album_date = songs_data[['album_id', 'publish_date']].drop_duplicates(
        subset=['album_id'], keep='first').reset_index(drop=True)
    album_df = album_id[['album_name', 'album_id']].merge(album_date,
                                                          on='album_id',
                                                          how='left')
    songs_data_cleared = songs_data.drop(['album_id', 'publish_date'],
                                         axis=1).copy()
    songs_data_cleared = songs_data_cleared.merge(album_df,
                                                  on='album_name',
                                                  how='left')
    return songs_data_cleared

# 歌词采集

In [11]:
def get_qq_lyric(song_id):
    """根据歌曲ID获取歌词"""
    url = "https://c.y.qq.com/lyric/fcgi-bin/fcg_query_lyric_yqq.fcg"
    params = {
        "nobase64": 1,
        "musicid": song_id,
        "format": "json"
    }
    headers = {
        "Referer": "https://y.qq.com/",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    response = requests.get(url, params=params, headers=headers)
    if response.status_code == 200:
        lyric_data = response.json()
        return lyric_data.get("lyric", "")
    return "歌词获取失败"

In [12]:
def get_all_songs_lyric(file_path, songs_df):
    for i, row in songs_df.iterrows():
        song_id = row['song_id']
        song_name = row['song_name']
        # 判断是否已采集
        if os.path.exists(file_path):
            df = pd.read_json(file_path)
            songs_had = df['song_id'].tolist()
            if song_id not in songs_had:
                print(song_name)
                lyric_raw = get_qq_lyric(song_id)
                single_res = {
                    'song_id': song_id,
                    'song_name': song_name,
                    'lyric_raw': lyric_raw
                }
                save_to_json_list(file_path, single_res)
                time.sleep(2)
        else:
            print(song_name)
            lyric_raw = get_qq_lyric(song_id)
            single_res = {
                    'song_id': str(song_id),
                    'song_name': song_name,
                    'lyric_raw': lyric_raw
                }
            save_to_json_list(file_path, single_res)
            time.sleep(2)

## 歌词清洗

In [13]:
class QQLyricCleanerV16:
    def __init__(self):
        self.time_tag_pattern = re.compile(r'\[(\d{2,}:?\d{2,}\.\d{2,})\]\s*(.*)')

    def _generate_singer_blacklist(self, singers):
        if not singers: return set()
        # 兼容中英文逗号、斜杠、空格拆分
        names = re.split(r'[,，/ ]', singers)
        blacklist = {'男', '女', '合', '人', '们'}
        for name in names:
            name = name.strip()
            if not name: continue
            blacklist.add(name.upper()) 
            if len(name) > 1:
                blacklist.add(name[0].upper()) # 姓
        return blacklist

    def _preprocess(self, text):
        if not text: return ""
        text = html.unescape(text)
        return text.replace('\r', '').replace('\\n', '\n')

    def get_credits(self, raw_text):
        text = self._preprocess(raw_text)
        res = {"lyricist": "", "composer": "", "arranger": ""}
        mapping = {
            "lyricist": r"(?:词|作词)\s*[:：]\s*([^\n\r\]]+)",
            "composer": r"(?:曲|作曲)\s*[:：]\s*([^\n\r\]]+)",
            "arranger": r"(?:编曲|制作人|Arranger|Program)\s*[:：]\s*([^\n\r\]]+)"
        }
        for key, pat in mapping.items():
            match = re.search(pat, text, re.IGNORECASE)
            if match:
                res[key] = re.split(r'[\[\]]', match.group(1).strip())[0].strip()
        return res

    def process_data(self, raw_lyric, song_id=None, song_name="", singers=""):
        singer_bits = self._generate_singer_blacklist(singers)
        credits = self.get_credits(raw_lyric)
        text = self._preprocess(raw_lyric)
        
        valid_lines = []
        lines = text.split('\n')
        content_line_idx = 0
        
        for line in lines:
            match = self.time_tag_pattern.search(line)
            if not match: continue
            
            ts, content = match.groups()
            content = content.strip()
            if not content: continue

            # --- 核心改进：标题/标题行判定 ---
            clean_content = content.replace(" ", "").upper()
            
            # 识别标题的特征：包含歌曲名、包含歌手名、或者包含大量分隔标点
            contains_singer = any(s in clean_content for s in singer_bits if len(s) > 1)
            is_metadata_structure = any(symbol in content for symbol in ['-', '《', '，', '(', '（'])
            
            # 判定：如果是前几行，且长得像标题或元数据，直接过滤
            if content_line_idx < 3:
                if (song_name and song_name.upper() in clean_content) or contains_singer or is_metadata_structure:
                    content_line_idx += 1
                    continue

            # 权利声明过滤
            if any(w in clean_content for w in ['权利保留', '未经许可', '著作权', '版权', '声明', '提供']):
                content_line_idx += 1
                continue

            # --- 冒号处理逻辑 ---
            if ':' in content or '：' in content:
                sep = ':' if ':' in content else '：'
                prefix, *suffix = content.split(sep, 1)
                prefix_clean = prefix.strip().upper()
                suffix_content = "".join(suffix).strip()

                if prefix_clean in singer_bits:
                    if suffix_content:
                        content = suffix_content
                    else:
                        content_line_idx += 1
                        continue
                else:
                    # 只要不在演唱者白名单里的冒号行，全部删除
                    content_line_idx += 1
                    continue

            # 最终清洗
            c_final = re.sub(r'\s+', '，', content).strip('，')
            if c_final:
                valid_lines.append((ts, c_final))
                content_line_idx += 1

        # 数据组装
        start_time = valid_lines[0][0] if valid_lines else ""
        lyrics_text = "。".join([x[1] for x in valid_lines])
        if lyrics_text: lyrics_text += "。"

        return {
            "song_id": song_id,
            "song_name": song_name,
            "start_time": start_time,
            "has_lyric": 1 if lyrics_text else 0,
            "lyricist": credits["lyricist"],
            "composer": credits["composer"],
            "arranger": credits["arranger"],
            "lyrics_text": lyrics_text
        }

## 清洗main

In [ ]:
def clear_and_save_lyric(path_prefix, songs_df):
    lyric_raw = pd.read_json(path_prefix+'raw_lyric_data.json')

    cleaner = QQLyricCleanerV16()
    res_list = []
    for i, row in songs_df.iterrows():
        song_id = row['song_id']
        song_name = row['song_name']
        singers = row.get('artist_name', "")
        lyric_raw_single = lyric_raw[lyric_raw['song_id'] == song_id]['lyric_raw'].values[0]
        single_res = cleaner.process_data(lyric_raw_single, song_id, song_name, singers)
        res_list.append(single_res)
    with open(path_prefix+'cleared_lyric_data.json', 'w', encoding='utf-8') as f:
        json.dump(res_list, f, ensure_ascii=False, indent=4)

# main

## 歌手-按专辑筛选数据

In [ ]:
# file_path_prefix = "data/jaychou/"
# singger = "周杰伦"
# max_page = 20

file_path_prefix = "data/mayday/"
singger = "五月天"
max_page = 20

### 曲目采集

In [ ]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singger, max_page=max_page)

In [ ]:
df_song_data_raw = pd.DataFrame(song_data_raw)
# 原始曲目保存
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [ ]:
# 重新读取数据
df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')

In [ ]:
# 曲目筛选，按专辑
# 周杰伦
album_list = [
    "Jay", "范特西", "八度空间", "叶惠美", "七里香", "十一月的萧邦", "依然范特西", "我很忙", "魔杰座", "跨时代", "惊叹号", "十二新作", "周杰伦的床边故事", "最伟大的作品", "哎呦，不错哦"
]
# 五月天
# album_list = [
#     '第一张创作专辑', '爱情万岁', '人生海海', '时光机', '神的孩子都在跳舞', '为爱而生', '后青春期的诗',
#     '第二人生（明日版）', '第二人生（末日版）', '自传', '知足 最真杰作选', '步步 自选作品辑 the Best of 1999-2013'
# ]
song_data_filted = filter_album_songs(singger, song_data_raw_read, album_list)

In [ ]:
len(song_data_filted)

### 数据验证
检查专辑中歌曲是否存在缺失，将缺失歌曲的数据手工添加到song_data_filted中

In [ ]:
song_data_cleared_1 = clear_song_data(song_data_filted, is_use_raw_song_name=False)
song_data_cleared_1

In [ ]:
song_data_cleared_1.groupby('album_name')['song_name'].count()

In [ ]:
# 问题专辑
# 周杰伦
album_list_to_fix = ['哎呦，不错哦']
# 五月天
# album_list_to_fix = ['第二人生（明日版）']
song_data_cleared_1[song_data_cleared_1['album_name'] == album_list_to_fix[0]]

In [ ]:
# 补充歌曲，并修改专辑信息
# 周杰伦
songs_dict_to_add = {'说好不哭 (with 五月天阿信)' : '最伟大的作品', '不爱我就拉倒': '最伟大的作品', 'Mojito': '最伟大的作品', '等你下课 (with 杨瑞代)': '最伟大的作品', '我是如此相信': '最伟大的作品', '英雄': '周杰伦的床边故事', '一路向北': '十一月的萧邦'}
# 五月天
# songs_dict_to_add = {'生命有一种绝对': '时光机', 'Enrich Your Life': '神的孩子都在跳舞', '垃圾车 (朋友版)': '神的孩子都在跳舞', '温柔 (还你自由版)': '知足 最真杰作选', '入阵曲': '步步 自选作品辑 the Best of 1999-2013', '离开地球表面': '步步 自选作品辑 the Best of 1999-2013', 'OAOA (丢掉名字性别)': '第二人生（末日版）', '我不愿让你一个人': '第二人生（末日版）'}
for i in song_data_raw_read:
    if i['song_name'] in songs_dict_to_add.keys():
        print(i)

In [ ]:
# 手工添加需要补全的歌曲
# 周杰伦
songs_to_add = [{
    'song_id': 105755384,
    'song_mid': '004Qscj80GYhGR',
    'song_name': '英雄',
    'song_subname': '《英雄联盟》中国品牌主题曲',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '周杰伦的床边故事',
    'album_id': 1306793,
    'album_mid': '001uJFiE0tbGGa',
    'duration': 200,
    'publish_time': 1458748800
}, {
    'song_id': 247261229,
    'song_mid': '001PLl3C4gPSCI',
    'song_name': '我是如此相信',
    'song_subname': '《天火》电影主题曲',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 9612009,
    'album_mid': '001hGx1Z0so1YX',
    'duration': 266,
    'publish_time': 1576339200
}, {
    'song_id': 268352018,
    'song_mid': '001glaI72k8BQX',
    'song_name': 'Mojito',
    'song_subname': '',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 12924001,
    'album_mid': '0009C3rp3Kfwg0',
    'duration': 185,
    'publish_time': 1591891200
}, {
    'song_id': 213922043,
    'song_mid': '0031TAKo0095np',
    'song_name': '不爱我就拉倒',
    'song_subname': '',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 4044657,
    'album_mid': '001CnPE31iJ899',
    'duration': 245,
    'publish_time': 1526313600
}, {
    'song_id': 212877900,
    'song_mid': '001J5QJL1pRQYB',
    'song_name': '等你下课 (with 杨瑞代)',
    'song_subname': '',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 3883404,
    'album_mid': '003bSL0v4bpKAx',
    'duration': 270,
    'publish_time': 1516204800
}, {
    'song_id': 237773700,
    'song_mid': '001qvvgF38HVc4',
    'song_name': '说好不哭 (with 五月天阿信)',
    'song_subname': '',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': '最伟大的作品',
    'album_id': 7876962,
    'album_mid': '002gBTVk4JEE2T',
    'duration': 222,
    'publish_time': 1568646000
}, {
    'song_id': 5105986,
    'song_mid': '001xd0HI0X9GNq',
    'song_name': '一路向北',
    'song_subname': '《头文字D》电影插曲',
    'artist_name': '周杰伦',
    'artist_id': 4558,
    'artist_mid': '0025NhlN2yWrP4',
    'album_name': 'J III MP3 Player',
    'album_id': 14311,
    'album_mid': '002MAeob3zLXwZ',
    'duration': 294,
    'publish_time': 1119542400
}]
# 修改专辑名称
songs_to_add_fixed = []
for song in songs_to_add:
    song['album_name'] = songs_dict_to_add[song['song_name']]
    songs_to_add_fixed.append(song)

In [ ]:
# 五月天
songs_to_add = [
    {
        'song_id': 405385,
        'song_mid': '001BAFqt1Ay4Vf',
        'song_name': '离开地球表面',
        'song_subname': '《开心超人》动画电影片尾曲',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '离开地球表面 Jump!',
        'album_id': 32775,
        'album_mid': '002PYDbl3I5L2k',
        'duration': 275,
        'publish_time': 1184860800
    },
    {
        'song_id': 4996096,
        'song_mid': '003Xy9E32vvMLe',
        'song_name': '入阵曲',
        'song_subname': '《兰陵王》电视剧主题曲',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '兰陵王 电视剧原声带',
        'album_id': 431765,
        'album_mid': '002adz882rV5uh',
        'duration': 209,
        'publish_time': 1377792000
    },
    {
        'song_id': 4932058,
        'song_mid': '003PaRAX3j5wJk',
        'song_name': '生命有一种绝对',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '摇滚本事 电影音乐原声带',
        'album_id': 96335,
        'album_mid': '0015r2I31enfaR',
        'duration': 239,
        'publish_time': 1038672000
    },
    {
        'song_id': 4830242,
        'song_mid': '000PoJAV4NPMzW',
        'song_name': '温柔 (还你自由版)',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '音乐电影-五月之恋',
        'album_id': 96361,
        'album_mid': '001ntd0y01uQ4g',
        'duration': 426,
        'publish_time': 1088611200
    },
    {
        'song_id': 1056504,
        'song_mid': '002Sv0dp3T3p5U',
        'song_name': 'OAOA (丢掉名字性别)',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '第二人生（末日版）',
        'album_id': 90142,
        'album_mid': '000IPRft1LSCqL',
        'duration': 282,
        'publish_time': 1323964800
    },
    {
        'song_id': 4834459,
        'song_mid': '002nqyCb1bUnk6',
        'song_name': 'Enrich Your Life',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': 'Enrich Your Life',
        'album_id': 62890,
        'album_mid': '0006oAnx03zXUC',
        'duration': 166,
        'publish_time': 1096560000
    },
    {
        'song_id': 4932456,
        'song_mid': '002qi7L00Cpb73',
        'song_name': '垃圾车 (朋友版)',
        'song_subname': '',
        'artist_name': '五月天',
        'artist_id': 74,
        'artist_mid': '000Sp0Bz4JXH0o',
        'album_name': '神的孩子都在跳舞',
        'album_id': 96368,
        'album_mid': '002plCgA0zOyYF',
        'duration': 243,
        'publish_time': 1099238400
    },
    {'song_id': 519403016, 'song_mid': '000B69Qg0S8WUF', 'song_name': '我不愿让你一个人', 'song_subname': '《今夜一起为爱鼓掌》电视剧插曲', 'artist_name': '五月天', 'artist_id': 74, 'artist_mid': '000Sp0Bz4JXH0o', 'album_name': '今夜一起为爱鼓掌 电视剧原声带', 'album_id': 56247634, 'album_mid': '001Jhk1t0SC1FZ', 'duration': 265, 'publish_time': 1727625600}

]
# 修改专辑名称
songs_to_add_fixed = []
for song in songs_to_add:
    song['album_name'] = songs_dict_to_add[song['song_name']]
    songs_to_add_fixed.append(song)

### 专辑名称手工修改

In [ ]:
# 五月天
album_to_fix_dict = {
    '第二人生（末日版）': '第二人生',
    '第二人生（明日版）': '第二人生',
    '步步 自选作品辑 the Best of 1999-2013': '步步 自选作品辑',
}

In [ ]:
song_data_filted.extend(songs_to_add_fixed)


In [ ]:
# 修正专辑名称，需要时运行
for i in song_data_filted:
    if i['album_name'] in album_to_fix_dict:
        i['album_name'] = album_to_fix_dict[i['album_name']]

In [ ]:
len(song_data_filted)

### 二次清洗

In [ ]:
# song_data_filted.extend(songs_to_add)
# 五月天歌曲名使用原始歌曲名，其他则为is_use_raw_song_name=False
song_data_cleared_2 = clear_song_data(song_data_filted, is_use_raw_song_name=False)
song_data_cleared_2

In [ ]:
song_data_cleared_2.groupby('album_name')['song_name'].count()

In [ ]:
song_data_cleared_2[song_data_cleared_2['album_name'] == '第二人生']

In [ ]:
# 需要手工删除的歌
songs_to_drop = [
    'T1 21 31 21 (Bonus Track)', '知足 (乐团版)', '拥抱 (2013新录制作品)',
    '温柔 (2013Remix版)', '憨人 (Live)'
]
# 删除song_data_cleared中song_name在songs_to_drop的行
song_data_cleared = song_data_cleared_2[~song_data_cleared_2['song_name'].
                                         isin(songs_to_drop)]
song_data_cleared

In [ ]:
song_data_cleared_final = clear_album_data(song_data_cleared_2)
song_data_cleared_final

In [ ]:
song_data_cleared_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

### 歌词采集

In [ ]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', song_data_cleared_final)

### 歌词清洗

In [ ]:
clear_and_save_lyric(file_path_prefix, song_data_cleared_final)

## 歌手-不需要按专辑筛选数据

In [ ]:
# file_path_prefix = "data/liuyuning/"
# singger = "刘宇宁"
# max_page = 14

file_path_prefix = "data/liyuchun/"
singer = "李宇春"
max_page = 6

### 曲目采集

In [ ]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singer, max_page=max_page)

In [ ]:
df_song_data_raw = pd.DataFrame(song_data_raw)
# 原始曲目保存
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [ ]:
# 重新读取数据
df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')

In [ ]:
df_song_data_raw_read

### 清洗

In [ ]:
df_songs = df_song_data_raw_read.copy()
# 歌曲数据
df_songs['song_name_unique'] = df_songs['song_name'].astype(str)
# 清洗song_name_unique，删除空格，将中文括号转为英文括号
df_songs['song_name_unique'] = df_songs['song_name_unique'].str.replace(' ', '')
df_songs['song_name_unique'] = df_songs['song_name_unique'].str.replace('（', '(').str.replace('）', ')')
# 删除括号内的内容
df_songs['song_name_pure'] = df_songs['song_name_unique'].str.replace(
    r'\(.*?\)', '', regex=True)
df_songs = df_songs.drop_duplicates(subset=['song_name_pure'], keep='first')
df_songs

In [ ]:
df_songs[df_songs['album_name_pure'].str.startswith('我们民谣')]

In [ ]:
df_songs['album_name_pure'] = df_songs['album_name'].astype(str).fillna('无')
df_song_filt_tv = df_songs[~df_songs['album_name_pure'].str.startswith('声生不息')]
df_song_filt_tv

In [ ]:
# 选择歌手独唱作品
df_song_filt_singer = df_song_filt_tv[df_song_filt_tv['artist_name'] == singer]
df_song_filt_singer = df_song_filt_singer.drop_duplicates(subset=['song_name_pure'], keep='first')
df_song_filt_singer

In [ ]:
df_song_filt_singer.groupby('album_name_pure')['song_name_pure'].count().sort_values()

In [ ]:
# 前120首
song_list = df_song_filt_singer.head(120)['song_name_pure'].to_list()
'/，'.join(song_list)

In [ ]:
df_song_filt_singer[df_song_filt_singer['song_name_pure'].str.startswith('我是你的')]

In [ ]:

df_songs

In [ ]:
# 选择歌手独唱作品
df_song = df_song_data_raw_read.copy()
df_song_filt_singer = df_song[df_song['artist_name'] == singer]
# 删除无专辑信息的作品
df_song_filt_album = df_song_filt_singer[df_song_filt_singer['album_name'].notna()]
# 删除song_name中括号中的内容
df_song_filt_album


In [ ]:
song_data_cleared = clear_song_data(song_data_filted,
                                    is_filter_ost=False,
                                    is_use_raw_song_name=True)
song_data_cleared

In [ ]:
song_data_cleared.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

# 李宇春

In [14]:
# file_path_prefix = "data/liuyuning/"
# singger = "刘宇宁"
# max_page = 14

file_path_prefix = "data/liyuchun/"
singer = "李宇春"
max_page = 14

### 曲目采集

In [ ]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singer, max_page=max_page)

In [ ]:
df_song_data_raw = pd.DataFrame(song_data_raw)
# 原始曲目保存
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [15]:
# 重新读取数据
df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')

In [16]:
df_song_data_raw_read

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time
0,641644435,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615,002ZOuVm3Qn20Y,清风醉,85081075,003yPfqX3jjXlP,235,1771466400
1,625008,001DIsVq3sSjGV,下个，路口，见,NaN,李宇春,4615,002ZOuVm3Qn20Y,Chris Lee 同名专辑,53014,004Z4sId156J79,210,1261411200
2,517385504,003SqJTl4fnhRC,大梦归离,《大梦归离》影视剧主题曲,李宇春,4615,002ZOuVm3Qn20Y,大梦归离 影视原声带,57721256,003up2WR1OtPci,281,1727233200
3,212992885,001TmXYt4B8W4d,蜀绣 (Live),NaN,李宇春,4615,002ZOuVm3Qn20Y,李宇春野蛮生长巡演LIVE自选,3899212,003IqTUf2cTV2s,307,1517328000
4,213468025,00467LeA1TSU69,1987我不知会遇见你,NaN,李宇春,4615,002ZOuVm3Qn20Y,李宇春2018流行（liú xíng）巡演LIVE辑,3975872,003UqEXI3NvbEl,332,1522512000
...,...,...,...,...,...,...,...,...,...,...,...,...
695,585852950,001Ipjw53RWUR0,珍惜 (DJ 阿若版),NaN,李宇春,0,0032fmHO2UDnV3,NaN,0,NaN,205,0
696,432748046,002OhRjd1Y67iI,李宇春-下个，路口，见 (晊恦 remix),NaN,晊恦,8662646,001VBJBX4QnKBW,NaN,0,NaN,290,0
697,428477333,003vbyKD3hSaUL,李宇春-下个路口见 (Fe₂O₃ remix),NaN,Fe₂O₃,7303176,000WEiWj4YwBJX,下个，路口，见,40823851,001CA4Te4LkZO7,227,1689782400
698,125428444,003eXKCb3tGOXV,HAPPY HOUR (2011武汉Why Me演唱会),NaN,李宇春,4615,002ZOuVm3Qn20Y,NaN,0,NaN,114,0


### 清洗

In [17]:
def clear_song_name(df):
    df = df.copy()
    # 歌曲数据
    df['song_name_unique'] = df['song_name'].astype(str)
    # 清洗song_name_unique，删除空格，将中文括号转为英文括号
    df['song_name_unique'] = df['song_name_unique'].str.replace(' ', '')
    df['song_name_unique'] = df['song_name_unique'].str.replace(
        '（', '(').str.replace('）', ')')
    # 删除括号内的内容
    df['song_name_pure'] = df['song_name_unique'].str.replace(r'\(.*?\)',
                                                              '',
                                                              regex=True)
    # 删除空格
    df['song_name_pure'] = df['song_name_pure'].str.replace(' ', '')

    return df

In [18]:
# 选择歌手独唱作品
def clear_song_singer(df, singer_name):
    df = df.copy()
    df = df[df['artist_name'] == singer_name]
    return df

In [19]:
# 删除疑似翻唱的作品，按专辑名筛选
def clear_song_tv_show(df, albums):
    df = df.copy()
    df['album_name_pure'] = df['album_name'].astype(str).fillna('无')
    for album in albums:
        df = df[~df['album_name_pure'].str.startswith(album)]
    return df

In [20]:
albums_to_delete = ['声生不息', '在吗']

In [21]:
df_songs = clear_song_name(df_song_data_raw_read)
df_songs = clear_song_singer(df_songs, singer)
df_songs = clear_song_tv_show(df_songs, albums_to_delete)
df_songs = df_songs.drop_duplicates(subset=['song_name_pure'], keep='first').reset_index(drop=True)
df_songs

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_unique,song_name_pure,album_name_pure
0,641644435,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615,002ZOuVm3Qn20Y,清风醉,85081075,003yPfqX3jjXlP,235,1771466400,清风醉,清风醉,清风醉
1,625008,001DIsVq3sSjGV,下个，路口，见,NaN,李宇春,4615,002ZOuVm3Qn20Y,Chris Lee 同名专辑,53014,004Z4sId156J79,210,1261411200,下个，路口，见,下个，路口，见,Chris Lee 同名专辑
2,517385504,003SqJTl4fnhRC,大梦归离,《大梦归离》影视剧主题曲,李宇春,4615,002ZOuVm3Qn20Y,大梦归离 影视原声带,57721256,003up2WR1OtPci,281,1727233200,大梦归离,大梦归离,大梦归离 影视原声带
3,212992885,001TmXYt4B8W4d,蜀绣 (Live),NaN,李宇春,4615,002ZOuVm3Qn20Y,李宇春野蛮生长巡演LIVE自选,3899212,003IqTUf2cTV2s,307,1517328000,蜀绣(Live),蜀绣,李宇春野蛮生长巡演LIVE自选
4,213468025,00467LeA1TSU69,1987我不知会遇见你,NaN,李宇春,4615,002ZOuVm3Qn20Y,李宇春2018流行（liú xíng）巡演LIVE辑,3975872,003UqEXI3NvbEl,332,1522512000,1987我不知会遇见你,1987我不知会遇见你,李宇春2018流行（liú xíng）巡演LIVE辑
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
295,125311603,000551fS08rSkq,Let’s get loud,NaN,李宇春,4615,002ZOuVm3Qn20Y,NaN,0,NaN,236,0,Let’sgetloud,Let’sgetloud,nan
296,125255669,002JVEAs1eAFRP,不可能的声音 (演唱会版),NaN,李宇春,4615,002ZOuVm3Qn20Y,NaN,0,NaN,366,0,不可能的声音(演唱会版),不可能的声音,nan
297,324649408,001BjOER0UzVsI,下个路口见(北京卫视春晚)现场版 11/02/03,NaN,李宇春,0,0032fmHO2UDnV3,NaN,0,NaN,172,1613259913,下个路口见(北京卫视春晚)现场版11/02/03,下个路口见现场版11/02/03,nan
298,125252256,003WSszq08MF1l,剪爱 (Live),NaN,李宇春,4615,002ZOuVm3Qn20Y,NaN,0,NaN,142,0,剪爱(Live),剪爱,nan


In [22]:
# 前130首
songs_list_all = df_songs['song_name_pure'].to_list()
songs_list_130 = df_songs.head(130)['song_name_pure'].to_list()


## 专辑数据
手工采集，这里没有用到

In [ ]:
# 遍历 file_path_prefix + album中的json文件，合并为1个列表
import os
import json
import glob

def merge_album_json_files(file_path_prefix):
    """
    遍历 file_path_prefix/album 目录中的所有 JSON 文件，合并为一个列表
    
    :param file_path_prefix: 文件路径前缀，例如 "data/liyuchun/"
    :return: 合并后的歌曲列表
    """
    album_dir = os.path.join(file_path_prefix, 'album')
    
    # 检查目录是否存在
    if not os.path.exists(album_dir):
        print(f"⚠ 目录不存在: {album_dir}")
        return []
    
    # 获取所有 JSON 文件
    json_files = glob.glob(os.path.join(album_dir, '*.json'))
    
    if not json_files:
        print(f"⚠ 在 {album_dir} 中没有找到 JSON 文件")
        return []
    
    print(f"✓ 找到 {len(json_files)} 个 JSON 文件")
    
    merged_songs = []
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
                
                # 处理不同的 JSON 结构
                if isinstance(data, list):
                    merged_songs.extend(data)
                    print(f"  ✓ {os.path.basename(json_file)}: {len(data)} 首歌曲")
                elif isinstance(data, dict):
                    # 如果是字典，尝试提取歌曲列表
                    if 'songs' in data:
                        merged_songs.extend(data['songs'])
                        print(f"  ✓ {os.path.basename(json_file)}: {len(data['songs'])} 首歌曲")
                    elif 'data' in data:
                        merged_songs.extend(data['data'])
                        print(f"  ✓ {os.path.basename(json_file)}: {len(data['data'])} 首歌曲")
                    else:
                        # 如果是单个歌曲对象，直接添加
                        merged_songs.append(data)
                        print(f"  ✓ {os.path.basename(json_file)}: 1 首歌曲（单个对象）")
                        
        except json.JSONDecodeError as e:
            print(f"  ✗ {os.path.basename(json_file)}: JSON 解析错误 - {e}")
        except Exception as e:
            print(f"  ✗ {os.path.basename(json_file)}: 读取错误 - {e}")
    
    print(f"\n✓ 合并完成，共 {len(merged_songs)} 首歌曲")
    return merged_songs


def extract_song_names(merged_songs):
    """
    从合并的歌曲列表中提取歌曲名称
    
    :param merged_songs: 合并后的歌曲列表
    :return: 歌曲名称列表
    """
    song_names = []
    
    for song in merged_songs:
        if isinstance(song, dict):
            # 尝试不同的字段名
            name = song.get('song_name') or song.get('songname') or song.get('name') or song.get('title')
            if name:
                song_names.append(name)
        elif isinstance(song, str):
            song_names.append(song)
    
    return song_names


# 使用示例
file_path_prefix = "data/liyuchun/"

# 合并所有专辑 JSON 文件
merged_songs = merge_album_json_files(file_path_prefix)


In [ ]:
album_songs_dict = []
for i in merged_songs:
    album_dict = {}
    album_dict['album_name'] = i['album_name']
    album_dict['song_list'] = [v['song_name'] for v in i['song_list']]
    album_songs_dict.append(album_dict)
album_songs_dict

In [ ]:
albums_date_dict = {'会跳舞的文艺青年':'2011-04-12',
 '再不疯狂我们就老了': '2012-09-04',
 '皇后与梦想':'2006-09-15',
 '流行':'2017-11-07',
 '少年中国':'2008-04-29',
 '李宇春（同名专辑）':'2009-12-22',
 '周末愉快': '2023-04-13',
 '哇':'2019-07-04',
 '野蛮生长（四张EP）':'2016-05-25',
 '1987我不知会遇见你':'2014-07-30',
 '我的':'2007-11-01'}

In [ ]:
album_songs_df = pd.DataFrame(album_songs_dict)
# 把song_list拆分到每一行
album_songs_df = album_songs_df.explode('song_list')
album_songs_df = album_songs_df.rename(columns={'song_list': 'song_name'})
album_songs_df['publish_date'] = album_songs_df['album_name'].map(
    albums_date_dict)
album_songs_df['song_name_pure'] = album_songs_df['song_name'].str.replace(
    r'\(.*?\)', '', regex=True)
# 删除空格
album_songs_df['song_name_pure'] = album_songs_df['song_name_pure'].str.replace(' ', '')
# 去重
album_songs_df = album_songs_df.sort_values(by='publish_date').reset_index(drop=True)
album_songs_df = album_songs_df.drop_duplicates(subset='song_name_pure', keep='first').reset_index(drop=True)
album_songs_df

## 歌单确认

In [42]:
songs_to_add = [
    '皇后与梦想', '下雨', '冰菊物语', '我的王国', '漂浮地铁', '今天有朵云爱我', '您所拨打的电话号码是空号',
    '一而再再而三地喜欢你', '人间乐园', 'TMD我爱你', '口音', '木兰', '开放'
]
songs_to_delete = ['今夜你会不会来', '春风十里', '情书', '那女孩对我说', '南方姑娘', '爱你所爱', '无心睡眠', '莫过于此', '天黑黑', '张三的歌', '漂洋过海来看你', '城里的月光', '不要对他说', '下个,路口,见']

In [43]:
# 添加
songs_list_final = songs_list_130.copy()
for i in songs_to_add:
    if i not in songs_list_final:
        print(i)
        songs_list_final.append(i)

今天有朵云爱我
TMD我爱你
木兰


In [44]:
# 删除
for i in songs_to_delete:
    if i in songs_list_final:
        print(i)
        songs_list_final.remove(i)

春风十里
那女孩对我说
爱你所爱
莫过于此
张三的歌
城里的月光
不要对他说
下个,路口,见


In [45]:
len(songs_list_final)

125

## 歌曲数据确认

In [51]:
df_songs_final = df_songs[df_songs['song_name_pure'].isin(
    songs_list_final)].reset_index(drop=True)

df_songs_final = df_songs_final.drop(columns=['song_name_unique'])
df_songs_final['song_name_unique'] = df_songs_final['song_name_pure']
df_songs_final['publish_date'] = df_songs_final['publish_time'].apply(
    lambda x: format_timestamp(x))
df_songs_final = df_songs_final[df_songs_final['publish_date'].str.contains('-')]
df_songs_final['publish_year'] = df_songs_final['publish_date'].apply(
    lambda x: x.split('-')[0])
df_songs_final

,song_id,song_mid,song_name,song_subname,artist_name,artist_id,artist_mid,album_name,album_id,album_mid,duration,publish_time,song_name_pure,album_name_pure,song_name_unique,publish_date,publish_year
0,641644435,001Qkqsw0oUIKX,清风醉,《江湖夜雨十年灯》电视剧主题曲/片尾曲,李宇春,4615,002ZOuVm3Qn20Y,清风醉,85081075,003yPfqX3jjXlP,235,1771466400,清风醉,清风醉,清风醉,2026-02-19,2026
1,625008,001DIsVq3sSjGV,下个，路口，见,NaN,李宇春,4615,002ZOuVm3Qn20Y,Chris Lee 同名专辑,53014,004Z4sId156J79,210,1261411200,下个，路口，见,Chris Lee 同名专辑,下个，路口，见,2009-12-22,2009
2,517385504,003SqJTl4fnhRC,大梦归离,《大梦归离》影视剧主题曲,李宇春,4615,002ZOuVm3Qn20Y,大梦归离 影视原声带,57721256,003up2WR1OtPci,281,1727233200,大梦归离,大梦归离 影视原声带,大梦归离,2024-09-25,2024
3,212992885,001TmXYt4B8W4d,蜀绣 (Live),NaN,李宇春,4615,002ZOuVm3Qn20Y,李宇春野蛮生长巡演LIVE自选,3899212,003IqTUf2cTV2s,307,1517328000,蜀绣,李宇春野蛮生长巡演LIVE自选,蜀绣,2018-01-31,2018
4,213468025,00467LeA1TSU69,1987我不知会遇见你,NaN,李宇春,4615,002ZOuVm3Qn20Y,李宇春2018流行（liú xíng）巡演LIVE辑,3975872,003UqEXI3NvbEl,332,1522512000,1987我不知会遇见你,李宇春2018流行（liú xíng）巡演LIVE辑,1987我不知会遇见你,2018-04-01,2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,105206640,001oOToq0IUkiM,爱有引力,NaN,李宇春,4615,002ZOuVm3Qn20Y,混蛋，我想你,1235613,000wzySP2ujpo3,221,1449676800,爱有引力,混蛋，我想你,爱有引力,2015-12-10,2015
121,410614,0033rZ6R11iy99,我的黑白色,NaN,李宇春,4615,002ZOuVm3Qn20Y,我的,33189,001SqLCp2RidK7,301,1193932800,我的黑白色,我的,我的黑白色,2007-11-02,2007
122,426544,004JYRZc0OGTCi,TMD我爱你,NaN,李宇春,4615,002ZOuVm3Qn20Y,超级女声 美梦成真,34687,001uSTif44XUoj,200,1210780800,TMD我爱你,超级女声 美梦成真,TMD我爱你,2008-05-15,2008
123,225559362,003SszVH4IlNqh,木兰,《王者荣耀》花木兰英雄主打歌,李宇春,4615,002ZOuVm3Qn20Y,天美十年典藏：全明星音乐特辑,4867924,00436LYE43MyXL,201,1540483200,木兰,天美十年典藏：全明星音乐特辑,木兰,2018-10-26,2018


In [52]:
df_songs_final.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

## 歌词采集

In [ ]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', df_songs_final)

## 歌词清洗

In [ ]:
clear_and_save_lyric(file_path_prefix, df_songs_final)

# 旧 歌词采集

In [ ]:
album_songs = album_songs_df['song_name_pure'].to_list()

In [ ]:
for i in songs_to_add:
    if i not in album_songs:
        album_songs.append(i)

In [ ]:
len(album_songs)

In [ ]:
df_songs['is_have'] = df_songs['song_name_pure'].isin(album_songs)
df_songs_01 = df_songs[df_songs['is_have'] == True]
song_list_01 = df_songs_01['song_name_pure'].tolist()
len(song_list_01)

In [ ]:
a = '清风醉/，下个，路口，见/，大梦归离/，蜀绣/，1987我不知会遇见你/，素人/，和你一样/，冬泳/，回春天/，无价之姐/，给女孩/，Hey你没错/，是你/，唐人街/，闭嘴跳舞/，再不疯狂我们就老了/，一而再再而三地喜欢你/，海上的月亮/，信/，珍惜/，周末愉快/，千年游/，咏春/，皇后与梦想/，银河中的星星/，如果我不是我/，隔空投送/，西门少年/，冬天快乐/，流行/，你是人间的四月天/，野蛮生长/，漂浮地铁/，山河故人/，无尽海/，存在感/，新物种/，软肋/，小宇宙/，五脏六腑/，潮水凶猛/，AOAEO出发/，漂洋过海来看你/，因为我们呼吸同样的空气/，1991的晚风/，面朝大海，春暖花开/，给给/，千域千寻/，夏天的电影/，爱得太傻/，少年中国/，动物世界/，那又怎样/，今天雨，可是我们在一起/，混蛋，我想你/，走进你的梦/，梨花香/，最浪漫的事/，丝萝/，纯真年代/，刀锋偏冷/，哇/，GoodGood/，MAGICALSHOW/，忽然之间/，你是如此难以忘记/，当时/，口音/，一百种热爱/，花容瘦/，一趟/，年轻气盛/，下个,路口,见/，David/，Privacy/，小朋友/，我的王国/，杏/，无常/，对不起，只是忽然很想你/，人间值得/，耳机/，蓝/，乘着风/，"鸵鸟逃跑了"/，我的心里只有你没有他/，闷/，譬如：你/，Hoodie/，秀才胡同/，若/，神回复/，我爱/，会跳舞的文艺青年/，好久不见/，无花果/，张三的歌/，APopSong/，清福/，IfYouWereTheOnlyGirlInTheWorld/，焰火/，常旅客/，开放/，ShakeIt/，想哭/，人间乐园/，为爱感动/，阿么/，你的甜蜜/，舞/，我变了/，你好吗我很好谢谢你呢/，一点一点/，下雨/，冰菊物语/，今天有朵云爱我/，您所拨打的电话号码是空号/，TMD我爱你'
a_l = a.split('/，')
a_l_01 = []
for i in a_l:
    if i not in song_list_01:
        song_list_01.append(i)
len(song_list_01)

In [ ]:
song_list_02 = []
for i in song_list_01:
    if i not in songs_to_delete:
        song_list_02.append(i)
len(song_list_02)

In [ ]:
for i in songs_to_delete:
    if i in album_songs:
        print(i)

### 歌词采集

In [ ]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', song_data_cleared)

### 歌词清洗

In [ ]:
song_data_cleared.info()

In [ ]:
clear_and_save_lyric(file_path_prefix, song_data_cleared)

# 关键词-采集

In [ ]:
# file_path_prefix = "data/liuyuning/"
# singger = "刘宇宁"
# max_page = 14

file_path_prefix = "data/newyear/"
singger = "过年"
max_page = 3

### 曲目采集

In [ ]:
# 原始曲目数据采集
song_data_raw = get_songs_data_raw(singger=singger, max_page=max_page)

In [ ]:
df_song_data_raw = pd.DataFrame(song_data_raw)
# 原始曲目保存
df_song_data_raw.to_csv(file_path_prefix + 'raw_song_data.csv', index=False)

In [ ]:
# 重新读取数据
df_song_data_raw_read = pd.read_csv(file_path_prefix + 'raw_song_data.csv')
song_data_raw_read = df_song_data_raw_read.to_dict(orient='records')

In [ ]:
song_data_raw_read

In [ ]:
song_data_cleared = clear_song_data(song_data_raw_read,
                                    is_filter_ost=False,
                                    is_use_raw_song_name=False)
song_data_cleared

In [ ]:
# 删除song_name_unique中有标点符号，有数字,的歌
def filter_song_name(song_name):
    # 如果歌名中有标点符号或数字，返回False
    if re.search(r'[^\w\s]', song_name) or re.search(r'\d', song_name):
        return False
    return True

song_data_cleared = song_data_cleared[song_data_cleared['song_name_unique'].apply(filter_song_name)]
# 按song_name_unique去重
song_data_cleared = song_data_cleared.drop_duplicates(subset=['song_name_unique'], keep='first')
song_data_cleared


In [ ]:
# 手动删除 东北民谣，银河系DISCO
song_data_cleared = song_data_cleared[~song_data_cleared['song_name_unique'].isin(['东北民谣', '银河系DISCO'])]
song_data_cleared

In [ ]:
song_data_cleared.to_csv(file_path_prefix + 'cleared_song_data.csv', index=False)

In [ ]:
aaa = """过年啦 过年啦

国泰民安中国年

我用成语拜新年


祝大家



一帆风顺二龙腾飞




三羊开泰四季平安




五福临门六六大顺




七星高照八方来财




吉星高照张灯结彩

辞旧迎新阖家欢乐

出口成章恭贺新禧

年年有余岁岁平安




一祝爷爷奶奶福如东海



二祝姥姥姥爷寿比南山



三祝爸爸妈妈万事如意



四祝叔叔阿姨好运连连



五祝老师同学意气风发



六祝自我超越勇往直前




啊 万象更新又一年

千言万语说不完

满怀憧憬共祝愿

祝愿大家团团圆圆

啊 万象更新又一年

千言万语说不完

满怀憧憬共祝愿

祝愿大家团团圆圆




吉星高照张灯结彩

辞旧迎新阖家欢乐

出口成章恭贺新禧

年年有余岁岁平安




一帆风顺二龙腾飞



三羊开泰四季平安



五福临门六六大顺



七星高照八方来财



九九同心十全十美



百尺竿头千里之志




万事如意 年复一年

万事如意年复一年

啊 万象更新又一年

千言万语说不完

满怀憧憬共祝愿

祝愿大家团团圆圆

啊 万象更新又一年

千言万语说不完

满怀憧憬共祝愿

祝愿大家团团圆圆"""

In [ ]:
# 删除aaa中的空格换为句号，文字间的首个换行符替换为句号，其他换行符删除
bbb = re.sub(r'\n+', '。', aaa.replace(' ', ''))

bbb

### 歌词采集

In [ ]:
get_all_songs_lyric(file_path_prefix+'raw_lyric_data.json', song_data_cleared)

### 歌词清洗

In [ ]:
song_data_cleared.info()

In [ ]:
clear_and_save_lyric(file_path_prefix, song_data_cleared)

# 采集后，修改模型后，重新处理数据

In [ ]:
file_path_prefix = "data/jaychou/"
# file_path_prefix = "data/mayday/"

In [ ]:
song_data_cleared_read = pd.read_csv(file_path_prefix +
                                     'cleared_song_data.csv')
clear_and_save_lyric(file_path_prefix, song_data_cleared_read)

In [ ]:
song_data_cleared_read